# Scribble Conditioning Comparison

We compare three scribble conditions, all using a neutral prompt:
- **Man scribble** — HED extracted from a man portrait
- **Woman scribble** — HED extracted from a woman portrait  
- **Averaged scribble** — pixel average of the two HED maps

The target distribution is 100 images (50 man + 50 woman) generated with specific gender prompts.
Each condition generates 100 images with a neutral prompt and we measure MMD vs the target.

## 1. Setup

In [ ]:
import sys
if 'google.colab' in str(get_ipython()):
    !pip install -q diffusers transformers accelerate controlnet_aux scikit-learn peft

    from huggingface_hub import login
    from google.colab import userdata
    import getpass, os

    hf_token = userdata.get('HF') if hasattr(userdata, 'get') else None
    login(token=hf_token) if hf_token else login()

    github_token = userdata.get('GITHUB') if hasattr(userdata, 'get') else getpass.getpass('GitHub token: ')
    repo_url  = f'https://{github_token}@github.com/orineo1/conditional-matching-paper.git'
    repo_name = 'conditional-matching-paper'
    branch    = 'compareSDvsNaive'

    if not os.path.exists(repo_name):
        !git clone {repo_url}
    else:
        !cd {repo_name} && git pull
    !cd {repo_name} && git checkout {branch}

    for path in [f'/content/{repo_name}', f'/content/{repo_name}/SD_cond_SD_controlnet']:
        if path not in sys.path:
            sys.path.insert(0, path)
    print(f'✅ Repo ready on branch: {branch}')

## 2. Imports

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.decomposition import PCA
from IPython.display import display

from models        import load_models
from image_utils   import sobel_proxy, build_base_image
from clip_utils    import load_clip_model, encode_images_clip
from generation    import generate_and_store_cs
from visualization import plot_row
from metrics       import compute_mmd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 3. Config

In [ ]:
# Portraits used to extract scribbles (one per gender is enough)
N_SOURCE = 10

# Target distribution size (split evenly man/woman)
N_TARGET = 250

# Conditional images generated per scribble condition
N_COND = 250

CONTROLNET_SCALE = 0.5

MAN_PROMPT    = 'a superrealistic portrait photograph of a man, studio lighting'
WOMAN_PROMPT  = 'a superrealistic portrait photograph of a woman, studio lighting'
NEUTRAL_PROMPT = 'a superrealistic professional photograph of'

print(f'N_TARGET={N_TARGET}  N_COND={N_COND}  controlnet_scale={CONTROLNET_SCALE}')

## 4. Load Models

In [ ]:
architect, sprinter = load_models(device)
clip_model, clip_processor = load_clip_model(device)
print('✅ Models loaded.')

## 5. Base Oval & Sobel Conditioning

In [ ]:
from IPython.display import display

base_image_pil, base_tensor = build_base_image(device)
with torch.no_grad():
    sobel_pil = T.ToPILImage()(sobel_proxy(base_tensor, device).squeeze(0).cpu())

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(base_image_pil); axes[0].set_title('Base Oval');          axes[0].axis('off')
axes[1].imshow(sobel_pil, cmap='gray'); axes[1].set_title('Sobel Cond'); axes[1].axis('off')
plt.tight_layout()
display(fig)
plt.close()

## 6. Generate Source Portraits (for Scribble Extraction)

In [ ]:


def generate_and_store_cs(pipe, prompt, cond_pil, num_samples, batch_size=2, cn_scale=0.5,seed=None):
    """generate_and_store but with configurable controlnet_conditioning_scale."""
    original_vae_dtype = pipe.vae.dtype
    pipe.vae.to(dtype=torch.float16)
    all_images, all_lats = [], []

    def latents_callback(p, step_index, timestep, cb_kwargs):
        if step_index == p.num_timesteps - 1:
            p._current_latents = cb_kwargs["latents"].detach().cpu().numpy()
        return cb_kwargs

    generator = None
    if seed is not None:
        generator = torch.Generator(device=pipe.device).manual_seed(seed)
    for i in range(0, num_samples, batch_size):
        curr   = min(batch_size, num_samples - i)
        result = pipe(
            prompt=[prompt] * curr,
            image=[cond_pil] * curr,
            num_inference_steps=2,
            guidance_scale=0.0,
            controlnet_conditioning_scale=cn_scale,
            callback_on_step_end=latents_callback,
            generator=generator,
        )
        all_images.extend(result.images)
        all_lats.append(pipe._current_latents.reshape(curr, -1))
        print(f"  Progress: {len(all_images)}/{num_samples}", end="\r")

    print()
    pipe.vae.to(dtype=original_vae_dtype)
    return all_images, np.vstack(all_lats)

In [ ]:
SEED = 0

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

# We only need one man and one woman portrait to extract scribbles from
print('Generating source portraits...')
with torch.no_grad():
    man_images, _   = generate_and_store_cs(sprinter, MAN_PROMPT,   sobel_pil, N_SOURCE, 2, cn_scale=CONTROLNET_SCALE, seed=SEED)
    woman_images, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, sobel_pil, N_SOURCE, 2, cn_scale=CONTROLNET_SCALE, seed=SEED + 1)

source_man   = man_images[0]
source_woman = woman_images[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(source_man);   axes[0].set_title('Source Man');   axes[0].axis('off')
axes[1].imshow(source_woman); axes[1].set_title('Source Woman'); axes[1].axis('off')
plt.suptitle('Source Portraits for Scribble Extraction', fontsize=13, fontweight='bold')
plt.tight_layout()
display(fig)
plt.close()

## 7. Extract HED Scribbles & Compute Average

In [ ]:
from controlnet_aux import HEDdetector
from IPython.display import display

hed = HEDdetector.from_pretrained('lllyasviel/Annotators')

scribble_man   = hed(source_man,   scribble=True)
scribble_woman = hed(source_woman, scribble=True)

# Pixel-average the two HED maps
avg_np       = ((np.array(scribble_man).astype(np.float32) +
                 np.array(scribble_woman).astype(np.float32)) / 2.0).clip(0, 255).astype(np.uint8)
scribble_avg = Image.fromarray(avg_np)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 0: source portraits
axes[0, 0].imshow(source_man);   axes[0, 0].set_title('Source Man',   fontsize=12); axes[0, 0].axis('off')
axes[0, 1].imshow(source_woman); axes[0, 1].set_title('Source Woman', fontsize=12); axes[0, 1].axis('off')
axes[0, 2].set_visible(False)

# Row 1: scribbles
axes[1, 0].imshow(scribble_man,   cmap='gray'); axes[1, 0].set_title('Scribble Man',        fontsize=12); axes[1, 0].axis('off')
axes[1, 1].imshow(scribble_woman, cmap='gray'); axes[1, 1].set_title('Scribble Woman',      fontsize=12); axes[1, 1].axis('off')
axes[1, 2].imshow(scribble_avg,   cmap='gray'); axes[1, 2].set_title('Scribble Avg (mean)', fontsize=12); axes[1, 2].axis('off')

plt.suptitle('HED Scribbles', fontsize=14, fontweight='bold')
plt.tight_layout()
display(fig)
plt.close()
print('✅ Scribbles ready.')

## 8. Generate Target Distribution
100 images with specific gender prompts — this is what we measure MMD against.

Note: targets are generated from the oval Sobel (neutral shape), not a face scribble,
so the target distribution is not biased toward any specific face geometry.

In [ ]:
from IPython.display import display

n_half = N_TARGET // 2

print(f'Generating {n_half} target man images...')
with torch.no_grad():
    target_man, _ = generate_and_store_cs(sprinter, MAN_PROMPT, sobel_pil, n_half, batch_size=2, cn_scale=CONTROLNET_SCALE)

print(f'Generating {n_half} target woman images...')
with torch.no_grad():
    target_woman, _ = generate_and_store_cs(sprinter, WOMAN_PROMPT, sobel_pil, n_half, batch_size=2, cn_scale=CONTROLNET_SCALE)

target_images = target_man + target_woman
print(f'✅ Target: {len(target_images)} images ({n_half} man + {n_half} woman)')

for imgs, title in [(target_man, f'Target Man (N={n_half})'), (target_woman, f'Target Woman (N={n_half})')]:
    fig, axes = plt.subplots(1, 8, figsize=(24, 3))
    fig.suptitle(title, fontsize=13, fontweight='bold')
    for ax, img in zip(axes, imgs[:8]):
        ax.imshow(img); ax.axis('off')
    for ax in axes[len(imgs[:8]):]:
        ax.axis('off')
    plt.tight_layout()
    display(fig)
    plt.close()

## 9. Encode Targets to CLIP

In [ ]:
print('Encoding target images to CLIP...')
target_tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in target_images], dim=0).to(device)

clip_model.to(device)
with torch.no_grad():
    target_clip_embs = encode_images_clip(target_tensors, clip_model, clip_processor)  # [100, 768]
clip_model.to('cpu')

print(f'Target CLIP embeddings: {target_clip_embs.shape}')

# Sanity check
intra = (target_clip_embs[:n_half] @ target_clip_embs[:n_half].T).mean().item()
inter = (target_clip_embs[:n_half] @ target_clip_embs[n_half:].T).mean().item()
print(f'Intra-class sim (man↔man):   {intra:.4f}')
print(f'Inter-class sim (man↔woman): {inter:.4f}')
assert intra > inter, 'Classes not separated!'
print('✅ Classes separable in CLIP space.')

## 10. Generate 100 Images per Condition (Neutral Prompt)

In [ ]:
conditions = {
    'man_scribble':   scribble_man,
    'woman_scribble': scribble_woman,
    'avg_scribble':   scribble_avg,
}

def generate_condition(scribble_pil, prompt, n, batch_size=2):
    original_dtype = sprinter.vae.dtype
    sprinter.vae.to(dtype=torch.float16)
    images = []
    with torch.no_grad():
        for start in range(0, n, batch_size):
            bs = min(batch_size, n - start)
            result = sprinter(
                prompt=[prompt] * bs,
                image=[scribble_pil] * bs,
                num_inference_steps=2,
                guidance_scale=0.0,
                controlnet_conditioning_scale=CONTROLNET_SCALE,
                output_type='pil',
            )
            images.extend(result.images)
            print(f'  {len(images)}/{n}', end='\r')
    sprinter.vae.to(dtype=original_dtype)
    print()
    return images

cond_images = {}
for name, scribble in conditions.items():
    print(f'\nGenerating {N_COND} images — {name}')
    cond_images[name] = generate_condition(scribble, NEUTRAL_PROMPT, N_COND)

print('✅ All conditions generated.')

## 11. Encode Conditional Images to CLIP

In [ ]:
print('Encoding conditional images to CLIP...')
cond_clip_embs = {}

clip_model.to(device)
for name, images in cond_images.items():
    tensors = torch.cat([TF.to_tensor(img).unsqueeze(0) for img in images], dim=0).to(device)
    with torch.no_grad():
        embs = encode_images_clip(tensors, clip_model, clip_processor)
    cond_clip_embs[name] = embs
    print(f'  {name}: {embs.shape}')
clip_model.to('cpu')

print('✅ Done.')

## 12. Compute MMD — Each Condition vs Target

In [ ]:
print(f'{"Condition":<20}  MMD')
print('-' * 32)

mmd_results = {}
for name, embs in cond_clip_embs.items():
    mmd = compute_mmd(embs, target_clip_embs.detach()).item()
    mmd_results[name] = mmd
    print(f'{name:<20}  {mmd:.6f}')

best = min(mmd_results, key=mmd_results.get)
print(f'\n✅ Best (lowest MMD): {best}  ({mmd_results[best]:.6f})')

## 13. PCA — Conditions vs Target Distribution

In [ ]:
from IPython.display import display

all_embs = torch.cat([target_clip_embs] + list(cond_clip_embs.values()), dim=0).cpu().numpy()
pca      = PCA(n_components=2)
coords   = pca.fit_transform(all_embs)

target_coords = coords[:N_TARGET]
offset        = N_TARGET

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(target_coords[:n_half, 0], target_coords[:n_half, 1], c='royalblue', alpha=0.4, s=30, label='Target man')
ax.scatter(target_coords[n_half:, 0], target_coords[n_half:, 1], c='crimson',   alpha=0.4, s=30, label='Target woman')

colors  = ['orange', 'purple', 'limegreen']
markers = ['s', '^', 'x']
for (name, _), color, marker in zip(cond_clip_embs.items(), colors, markers):
    c = coords[offset:offset + N_COND]
    ax.scatter(c[:, 0], c[:, 1], c=color, alpha=0.5, s=40, marker=marker,
               label=f'{name}  (MMD={mmd_results[name]:.4f})')
    offset += N_COND

ax.set_title(f'CLIP PCA — Conditions vs Target\nVariance explained: {pca.explained_variance_ratio_.sum():.1%}', fontsize=13)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
display(fig)
plt.close()

## 14. Visual Grid — Sample Images per Condition

In [ ]:
from IPython.display import display

N_SHOW = 8

n_rows = 1 + len(conditions)
n_cols = 1 + N_SHOW

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.5 * n_cols, 3 * n_rows))

# Row 0: target samples
axes[0, 0].imshow(sobel_pil, cmap='gray')
axes[0, 0].set_title('Cond', fontsize=8)
axes[0, 0].set_ylabel('Target\ndistribution', fontsize=9, rotation=0, labelpad=65, va='center')
axes[0, 0].axis('off')
for j, img in enumerate(target_images[:N_SHOW]):
    axes[0, j + 1].imshow(img); axes[0, j + 1].axis('off')

# Rows 1-3: conditions
scribble_list = [scribble_man, scribble_woman, scribble_avg]
for row, (name, scribble) in enumerate(zip(cond_images.keys(), scribble_list), start=1):
    axes[row, 0].imshow(scribble, cmap='gray')
    axes[row, 0].set_title('Cond', fontsize=8)
    axes[row, 0].set_ylabel(f'{name}\nMMD={mmd_results[name]:.4f}',
                             fontsize=9, rotation=0, labelpad=65, va='center')
    axes[row, 0].axis('off')
    for j, img in enumerate(cond_images[name][:N_SHOW]):
        axes[row, j + 1].imshow(img); axes[row, j + 1].axis('off')

plt.suptitle(
    f'Neutral prompt: "{NEUTRAL_PROMPT}"\n'
    f'N={N_COND} images per condition — MMD vs {N_TARGET} targets',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
display(fig)
plt.close()

In [ ]:
from run_dps import compute_clip_softmax
from IPython.display import display

# ── Collect results for all conditions ────────────────────────────────────────
all_results = {}
for name in conditions.keys():
    results, _ = compute_clip_softmax(
        cond_images[name], clip_model, clip_processor,
        MAN_PROMPT, WOMAN_PROMPT, device
    )
    all_results[name] = results

# ── Per-condition: top-5 men / top-5 women grid ───────────────────────────────
for name, results in all_results.items():
    mmd_val  = mmd_results[name]
    n_male   = sum(1 for r in results if r["label"] == "male")
    n_female = sum(1 for r in results if r["label"] == "female")
    print(f"{name} — Male: {n_male}  Female: {n_female}  MMD: {mmd_val:.4f}")

    paired    = list(zip(cond_images[name], results))
    top_men   = sorted([(img, r) for img, r in paired if r["label"] == "male"],
                       key=lambda x: x[1]["p_male"],   reverse=True)[:5]
    top_women = sorted([(img, r) for img, r in paired if r["label"] == "female"],
                       key=lambda x: x[1]["p_female"], reverse=True)[:5]

    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for col, (img, r) in enumerate(top_men):
        axes[0, col].imshow(img)
        axes[0, col].set_title(f'p={r["p_male"]:.2f}', fontsize=9, color='royalblue')
        axes[0, col].axis('off')
    for col in range(len(top_men), 5):
        axes[0, col].axis('off')
    for col, (img, r) in enumerate(top_women):
        axes[1, col].imshow(img)
        axes[1, col].set_title(f'p={r["p_female"]:.2f}', fontsize=9, color='crimson')
        axes[1, col].axis('off')
    for col in range(len(top_women), 5):
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel('Top 5 Men',   fontsize=11, color='royalblue', labelpad=8)
    axes[1, 0].set_ylabel('Top 5 Women', fontsize=11, color='crimson',   labelpad=8)
    fig.suptitle(f"{name} — {n_male}M / {n_female}F  |  MMD={mmd_val:.4f}",
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    display(fig)
    plt.close()

# ── Combined boxplot: all conditions, x=gender, y=confidence ─────────────────
fig, axes = plt.subplots(1, len(all_results), figsize=(5 * len(all_results), 5), sharey=True)

for ax, (name, results) in zip(axes, all_results.items()):
    male_conf   = [r["p_male"]   for r in results if r["label"] == "male"]
    female_conf = [r["p_female"] for r in results if r["label"] == "female"]

    bp = ax.boxplot(
        [male_conf if male_conf else [float('nan')],
         female_conf if female_conf else [float('nan')]],
        labels=[f'Male\n(n={len(male_conf)})', f'Female\n(n={len(female_conf)})'],
        patch_artist=True,
        medianprops=dict(color='black', linewidth=2),
    )
    bp['boxes'][0].set_facecolor('royalblue'); bp['boxes'][0].set_alpha(0.6)
    bp['boxes'][1].set_facecolor('crimson');   bp['boxes'][1].set_alpha(0.6)

    # overlay individual points
    for i, (scores, color) in enumerate([(male_conf, 'royalblue'), (female_conf, 'crimson')], start=1):
        ax.scatter([i] * len(scores), scores, color=color, alpha=0.5, s=30, zorder=3)

    ax.set_title(f'{name}\nMMD={mmd_results[name]:.4f}', fontsize=11, fontweight='bold')
    ax.set_ylim(0.5, 1.0)
    ax.grid(True, alpha=0.3, axis='y')

axes[0].set_ylabel('Confidence', fontsize=11)
fig.suptitle('CLIP Softmax Confidence by Gender — All Conditions',
             fontsize=13, fontweight='bold')
plt.tight_layout()
display(fig)
plt.close()